# Gambaran buat training nanti

### alasan kenapa ga perlu pisah file per komoditas

datanya udah dalam long format. jadi selama kita selalu groupby('Komoditas') saat membuat fitur berbasis waktu, urutan waktu tiap komoditas tetap terjaga dan ga ada kebocoran antar komoditas. Jadi satu file aja udah cukup, pemisahan per kelompok cuman perlu dilakukan pas mau training, gaperlu dipisah dari awal.

In [9]:
import pandas as pd
import numpy as np

In [10]:
df = pd.read_csv('data/cleaned/pihps_featured.csv', parse_dates=['tanggal'])
df = df.sort_values(['Komoditas', 'tanggal']).reset_index(drop=True)

print(df.shape)
df.head()

(23709, 11)


,Komoditas,tanggal,harga,sumber,is_outlier,is_holiday,is_ramadan,days_to_lebaran,suhu_rata2,curah_hujan,kecepatan_angin
0,Bawang Merah Ukuran Sedang,2022-01-03,30200.0,PIHPS,False,0,0,-119,27.4,0.5,10.6
1,Bawang Merah Ukuran Sedang,2022-01-04,30150.0,PIHPS,False,0,0,-118,27.4,0.0,10.2
2,Bawang Merah Ukuran Sedang,2022-01-05,30300.0,PIHPS,False,0,0,-117,26.4,13.0,8.9
3,Bawang Merah Ukuran Sedang,2022-01-06,30300.0,PIHPS,False,0,0,-116,26.2,3.6,8.6
4,Bawang Merah Ukuran Sedang,2022-01-07,30400.0,PIHPS,False,0,0,-115,26.9,4.3,16.1


## Kelompok Komoditas

dari EDA yang kulakukan, komoditas dalam satu kelompok berkorelasi sangat tinggi (>0.95) dan punya karakter harga yang serupa. makanya satu model per kelompok lebih masuk akal karena lebih efisien daripada 21 model terpisah, dan lebih akurat dari satu model untuk semua 21 komoditas sekaligus ( karena skala harganya terlalu beda, daging sapi vs gula misalnya).

dict di bawah ini bisa dipake buat menentukan komoditas mana masuk kelompok mana. pemisahanya pas difilter nanti.

In [11]:
KELOMPOK = {
    'Beras':         [c for c in df['Komoditas'].unique() if 'Beras' in c],
    'Cabai':         [c for c in df['Komoditas'].unique() if 'Cabai' in c],
    'Daging_Telur':  [c for c in df['Komoditas'].unique() if 'Daging' in c or 'Telur' in c],
    'Bawang':        [c for c in df['Komoditas'].unique() if 'Bawang' in c],
    'Minyak_Goreng': [c for c in df['Komoditas'].unique() if 'Minyak' in c],
    'Gula':          [c for c in df['Komoditas'].unique() if 'Gula' in c],
}

for nama, anggota in KELOMPOK.items():
    print(f'{nama}: {anggota}')

Beras: ['Beras Kualitas Bawah I', 'Beras Kualitas Bawah II', 'Beras Kualitas Medium I', 'Beras Kualitas Medium II', 'Beras Kualitas Super I', 'Beras Kualitas Super II']
Cabai: ['Cabai Merah Besar', 'Cabai Merah Keriting', 'Cabai Rawit Hijau', 'Cabai Rawit Merah']
Daging_Telur: ['Daging Ayam Ras Segar', 'Daging Sapi Kualitas 1', 'Daging Sapi Kualitas 2', 'Telur Ayam Ras Segar']
Bawang: ['Bawang Merah Ukuran Sedang', 'Bawang Putih Ukuran Sedang']
Minyak_Goreng: ['Minyak Goreng Curah', 'Minyak Goreng Kemasan Bermerk 1', 'Minyak Goreng Kemasan Bermerk 2']
Gula: ['Gula Pasir Kualitas Premium', 'Gula Pasir Lokal']


## Lag Features

karena model time series ga bisa hanya lihat kondisi hari ini misalnya faktor ramadan, cuaca, dll. tanpa tahu harga sebelumnya. Lag features ini adalah harga masa lalu yang bisa dijadikan input, misalnya seperti untuk mencari tahu "berapa harga komoditas ini 1 hari lalu, 7 hari lalu, 14 hari lalu, 30 hari lalu?"

rolling mean juga bisa dipake buat melengkapi dengan gambaran tren rata-rata jangka pendek dan menengah. `shift(1)` sebelum rolling idealnya rata-rata yang dihitung ga menyertakan harga hari ini, biar ga bocor nilai yang mau diprediksi ke dalam fiturnya.

semuanya dihitung dengan groupby('Komoditas') supaya shift ga melompat antar komoditas.

In [12]:
for lag in [1, 7, 14, 30]:
    df[f'harga_lag_{lag}'] = df.groupby('Komoditas')['harga'].shift(lag)

for window in [7, 14, 30]:
    df[f'rolling_mean_{window}'] = (
        df.groupby('Komoditas')['harga']
          .transform(lambda x: x.shift(1).rolling(window).mean())
    )

# Cek hasilnya untuk satu komoditas
df[df['Komoditas'] == 'Cabai Rawit Merah'][
    ['tanggal', 'harga', 'harga_lag_1', 'harga_lag_7', 'rolling_mean_7']
].head(12)

,tanggal,harga,harga_lag_1,harga_lag_7,rolling_mean_7
12419,2022-01-03,86700.0,NaN,NaN,NaN
12420,2022-01-04,85200.0,86700.0,NaN,NaN
12421,2022-01-05,81800.0,85200.0,NaN,NaN
12422,2022-01-06,77200.0,81800.0,NaN,NaN
12423,2022-01-07,74500.0,77200.0,NaN,NaN
12424,2022-01-10,67500.0,74500.0,NaN,NaN
12425,2022-01-11,65450.0,67500.0,NaN,NaN
12426,2022-01-12,64350.0,65450.0,86700.0,76907.142857
12427,2022-01-13,63500.0,64350.0,85200.0,73714.285714
12428,2022-01-14,62100.0,63500.0,81800.0,70614.285714


NaN di baris pertama itu normal, soalnya baris paling awal ga punya data historis untuk dipakai sebagai lag, Baris-baris ini perlu didrop saat dropna() nanti sebelum training.

## Encode Komoditas

nama komoditas perlu diubah ke angka karena model ga bisa baca string. Dengan ini, satu model dalam satu kelompok bisa membedakan antar varian, misalnya Cabai Rawit Merah vs Cabai Rawit Hijau hanya dari encode.

In [13]:
df['komoditas_encoded'] = df['Komoditas'].astype('category').cat.codes

# Simpan mapping-nya untuk keperluan decode saat inference
mapping = dict(enumerate(df['Komoditas'].astype('category').cat.categories))
print(mapping)

{0: 'Bawang Merah Ukuran Sedang', 1: 'Bawang Putih Ukuran Sedang', 2: 'Beras Kualitas Bawah I', 3: 'Beras Kualitas Bawah II', 4: 'Beras Kualitas Medium I', 5: 'Beras Kualitas Medium II', 6: 'Beras Kualitas Super I', 7: 'Beras Kualitas Super II', 8: 'Cabai Merah Besar', 9: 'Cabai Merah Keriting', 10: 'Cabai Rawit Hijau', 11: 'Cabai Rawit Merah', 12: 'Daging Ayam Ras Segar', 13: 'Daging Sapi Kualitas 1', 14: 'Daging Sapi Kualitas 2', 15: 'Gula Pasir Kualitas Premium', 16: 'Gula Pasir Lokal', 17: 'Minyak Goreng Curah', 18: 'Minyak Goreng Kemasan Bermerk 1', 19: 'Minyak Goreng Kemasan Bermerk 2', 20: 'Telur Ayam Ras Segar'}


## Fitur yang Dipakai

jadi kira2 nanti ini semua fiturnyya:

- **Lag & rolling**: sebagai konteks historis harga
- **is_holiday, is_ramadan, days_to_lebaran**: pola musiman dan kalender, sudah ada di `pihps_featured.csv`
- **suhu_rata2, curah_hujan, kecepatan_angin**: cuaca, relevan misalnya seperti untuk komoditas pertanian
- **komoditas_encoded**: supaya satu model bisa bedakan varian dalam kelompok yang sama

In [14]:
FEATURES = [
    'komoditas_encoded',
    'harga_lag_1', 'harga_lag_7', 'harga_lag_14', 'harga_lag_30',
    'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30',
    'is_holiday', 'is_ramadan', 'days_to_lebaran',
    'suhu_rata2', 'curah_hujan', 'kecepatan_angin',
]

TARGET = 'harga'

## Train/Test Split per Kelompok

splitnya nanti berbasis waktu dan bukan random, Kalau random nanti model bisa bocor melihat masa depan saat training.

fungsi di bawah ini yang handle semuanya: filter kelompok, drop baris NaN dari lag, lalu potong 3 bulan terakhir sebagai test set misalnya, makanya dataset ga perlu dipisah dari awal, filter ini yang bisa dipake.

In [15]:
def get_train_test(kelompok_name, test_months=3):
    data   = df[df['Komoditas'].isin(KELOMPOK[kelompok_name])].dropna(subset=FEATURES)
    cutoff = data['tanggal'].max() - pd.DateOffset(months=test_months)

    train = data[data['tanggal'] <= cutoff]
    test  = data[data['tanggal'] >  cutoff]

    print(f'{kelompok_name}: train {train.shape[0]} baris, test {test.shape[0]} baris')
    return train[FEATURES], train[TARGET], test[FEATURES], test[TARGET]


for nama in KELOMPOK:
    get_train_test(nama)

Beras: train 6210 baris, test 384 baris
Cabai: train 4140 baris, test 256 baris
Daging_Telur: train 4140 baris, test 256 baris
Bawang: train 2070 baris, test 128 baris
Minyak_Goreng: train 3105 baris, test 192 baris
Gula: train 2070 baris, test 128 baris


## Catatan

nah jadi nanti alur training per kelompok cukup tinggal perlu kek dibawah ini:

```python
for nama in KELOMPOK:
    X_train, y_train, X_test, y_test = get_train_test(nama)
    # fit model, evaluate, simpan
```

dan misalnya ntuk prediksi ke depan dan bukan hanya hari berikutnya, target juga perlu dishift negatif sesuai horizon kyk dibawah:

```python
horizon = 7  # prediksi 7 hari ke depan
df['target'] = df.groupby('Komoditas')['harga'].shift(-horizon)
```

baris paling akhir per komoditas akan jadi NaN dan perlu didrop juga sebelum training.